# 03 — Delayed-feedback contextual bandit simulation

The fixed score model supplies calibrated probabilities. The bandit alone adapts from released feedback. The simulator uses labels for costs internally but never sends a pending or permanently censored label to the closed-loop learner. Start with LightGBM; expand only after its protocol checks pass.

In [ ]:
from pathlib import Path
import json, os, sys
candidate_src = [Path(os.environ.get('FRAUDLAB_SRC', 'src')), Path('/kaggle/input/ieee-cis-fraud-code/src')]
for src in candidate_src:
    if (src / 'fraudlab').is_dir():
        sys.path.insert(0, str(src.resolve()))
        break
else:
    raise FileNotFoundError('Set FRAUDLAB_SRC to the src directory from this project.')

import pandas as pd
from fraudlab.config import ExperimentConfig, FeedbackConfig
from fraudlab.metrics import classification_metrics
from fraudlab.pipeline import load_simulation_input
from fraudlab.paths import resolve_artifact_dir
from fraudlab.simulator import run_bandit_simulation, run_threshold_baselines, save_simulation_outputs

FEATURE_ROOT = resolve_artifact_dir('features/ieee_cis_features.parquet')
EXPERIMENT_ROOT = resolve_artifact_dir('experiments/selected_feature_recipe.json')
OUTPUT_DIR = Path(os.environ.get('IEEE_CIS_OUTPUT_DIR', '/kaggle/working/artifacts' if Path('/kaggle/working').exists() else 'artifacts')) / 'simulation'
EXPERIMENT_DIR = EXPERIMENT_ROOT / 'experiments'
SELECTED_RECIPE = json.loads((EXPERIMENT_DIR / 'selected_feature_recipe.json').read_text())['recipe']
MODEL_NAMES = ['lightgbm']  # Expand only after LightGBM checks pass
RUN_SENSITIVITY = False  # Turn on after the base LightGBM pipeline passes
config = ExperimentConfig()
all_metrics = []
sensitivity_configs = {'base': config.feedback}
if RUN_SENSITIVITY:
    sensitivity_configs.update({
        'faster': FeedbackConfig(review_delay_days=1, maturity_days=7, fraud_delay_min_days=1, fraud_delay_max_days=7, fraud_censor_probability=0.20, seed=config.random_seed),
        'slower': FeedbackConfig(review_delay_days=1, maturity_days=60, fraud_delay_min_days=1, fraud_delay_max_days=60, fraud_censor_probability=0.20, seed=config.random_seed),
        'more_censoring': FeedbackConfig(review_delay_days=1, maturity_days=30, fraud_delay_min_days=1, fraud_delay_max_days=30, fraud_censor_probability=0.50, seed=config.random_seed),
    })

In [ ]:
for model_name in MODEL_NAMES:
    RECIPE_DIR = EXPERIMENT_DIR / 'recipes' / SELECTED_RECIPE
    prediction_path = RECIPE_DIR / 'predictions' / f'{model_name}_simulation_predictions.parquet'
    rows, probabilities = load_simulation_input(
        FEATURE_ROOT / 'features' / 'ieee_cis_features.parquet', prediction_path, config
    )
    threshold_config = json.loads((RECIPE_DIR / f'{model_name}_thresholds.json').read_text())
    baselines = run_threshold_baselines(rows, probabilities, config.costs, threshold_config)
    diagnostic_metrics = classification_metrics(rows['isFraud'].to_numpy(), probabilities)
    for scenario_name, feedback_config in sensitivity_configs.items():
        for mode in ('closed_loop', 'oracle'):
            decisions, metrics = run_bandit_simulation(
                rows, probabilities, mode, config.costs, feedback_config, config.bandit, seed=config.random_seed
            )
            metrics['model'] = model_name
            metrics['scenario'] = scenario_name
            metrics['oracle_tax'] = None
            save_simulation_outputs(decisions, metrics, OUTPUT_DIR, f'{model_name}_{scenario_name}_{mode}')
            all_metrics.append({'model': model_name, 'experiment': f'{mode}_{scenario_name}', **metrics, **diagnostic_metrics})
    for baseline_name, baseline_metrics in baselines.items():
        all_metrics.append({'model': model_name, 'experiment': baseline_name, **baseline_metrics, **diagnostic_metrics})

metrics_frame = pd.DataFrame(all_metrics)
for model_name in MODEL_NAMES:
    for scenario_name in sensitivity_configs:
        closed_name = f'closed_loop_{scenario_name}'
        oracle_name = f'oracle_{scenario_name}'
        closed = metrics_frame.loc[(metrics_frame['model'] == model_name) & (metrics_frame['experiment'] == closed_name), 'total_cost'].iloc[0]
        oracle = metrics_frame.loc[(metrics_frame['model'] == model_name) & (metrics_frame['experiment'] == oracle_name), 'total_cost'].iloc[0]
        metrics_frame.loc[(metrics_frame.model == model_name) & (metrics_frame.experiment.isin([closed_name, oracle_name])), 'oracle_tax'] = closed - oracle
    model_rows = metrics_frame.model == model_name
    baseline_costs = {name: metrics_frame.loc[model_rows & (metrics_frame.experiment == name), 'total_cost'].iloc[0] for name in ['always_approve', 'always_block', 'no_review', 'validation_tuned_thresholds']}
    for baseline_name, baseline_cost in baseline_costs.items():
        metrics_frame.loc[model_rows, f'cost_reduction_vs_{baseline_name}_pct'] = 100 * (baseline_cost - metrics_frame.loc[model_rows, 'total_cost']) / max(baseline_cost, 1e-9)
    metrics_frame.loc[model_rows, 'historical_phase1_cost'] = config.historical_phase1_cost
    metrics_frame.loc[model_rows, 'cost_reduction_vs_historical_phase1_pct'] = 100 * (config.historical_phase1_cost - metrics_frame.loc[model_rows, 'total_cost']) / config.historical_phase1_cost
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics_frame.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
display(metrics_frame[['model', 'experiment', 'total_cost', 'oracle_tax', 'review_rate', 'review_cap_utilization', 'pr_auc', 'precision_at_0_5', 'recall_at_0_5']])

In [ ]:
# Optional comparison plots; run after metrics.csv exists.
from fraudlab.plots import plot_cost_comparison, plot_detection_comparison
plot_cost_comparison(OUTPUT_DIR / 'metrics.csv', OUTPUT_DIR / 'cost_comparison.png')
plot_detection_comparison(OUTPUT_DIR / 'metrics.csv', OUTPUT_DIR / 'detection_comparison.png')